# Database Management with Python — SQLite3 Demo
**Purpose:** Demonstrate relational database concepts and everyday SQL operations from beginner to intermediate level.

This notebook builds a small college database with students, courses, instructors, and enrollments. It covers table design, primary/foreign keys, CRUD, filtering, sorting, aggregation, joins, subqueries, transactions, constraints, and parameterized queries.

**How to use**
1. Run cells from top to bottom.
2. The database file `college_demo.db` is created in the notebook's current working directory.
3. Re-running the setup cell resets the demo tables and sample data, so it is safe to start over.
4. Use the displayed query results as screenshots for your slides.

## 1. How a database works

A database is an organized collection of data. A **DBMS** (Database Management System) is software that stores, retrieves, updates, and protects that data. SQLite is a relational DBMS embedded in an application: it stores the database in a local file and does not require a separate database server.

Typical flow:

`Python application → sqlite3 driver → SQL statement → SQLite engine → database file`

- **Table:** a set of related records, arranged in rows and columns.
- **Row / record:** one item, such as one student.
- **Column / field:** one attribute, such as a student's name.
- **Schema:** the structure of the database—tables, columns, data types, constraints, and relationships.
- **Query:** a request to retrieve or modify data, usually written in SQL.
- **Result set:** the rows returned by a query.

## 2. Relational design and our example

We will model a small college system:

- **Students**: one row per student.
- **Courses**: one row per course.
- **Instructors**: one row per instructor.
- **Enrollments**: connects students to courses and stores semester/grade information.

Relationship design:
- One instructor can teach many courses (**one-to-many**).
- A student can take many courses, and a course can have many students (**many-to-many**), represented by the `enrollments` junction table.
- Primary keys uniquely identify rows.
- Foreign keys refer to rows in another table.

In [26]:
import sqlite3
from pathlib import Path
from pprint import pprint

DB_PATH = Path("college_demo.db")
print("Database file:", DB_PATH.resolve())

Database file: /content/college_demo.db


## 3. Connect to SQLite

`sqlite3.connect()` opens an existing database file or creates it if it does not exist. `conn.cursor()` creates a cursor, which executes SQL and fetches results.

SQLite foreign-key enforcement is enabled per connection, so we explicitly enable it with `PRAGMA foreign_keys = ON`.

In [27]:
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")
cur = conn.cursor()

print("SQLite version:", sqlite3.sqlite_version)
print("Foreign keys enabled:", conn.execute("PRAGMA foreign_keys").fetchone()[0])

SQLite version: 3.45.1
Foreign keys enabled: 1


## 4. Create the schema

`CREATE TABLE` defines a table. Common constraints:
- `PRIMARY KEY`: unique identifier for each row.
- `NOT NULL`: a value is required.
- `UNIQUE`: duplicate values are not allowed in the constrained column(s).
- `CHECK`: a value must satisfy a condition.
- `FOREIGN KEY`: a value must refer to a valid row in another table.

In [28]:
# Reset tables so the notebook can be rerun from the beginning.
cur.executescript("""
DROP TABLE IF EXISTS enrollments;
DROP TABLE IF EXISTS courses;
DROP TABLE IF EXISTS students;
DROP TABLE IF EXISTS instructors;

CREATE TABLE instructors (
    instructor_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    department TEXT NOT NULL
);

CREATE TABLE students (
    student_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    age INTEGER CHECK (age >= 16),
    city TEXT NOT NULL
);

CREATE TABLE courses (
    course_id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    credits INTEGER NOT NULL CHECK (credits BETWEEN 1 AND 6),
    instructor_id INTEGER NOT NULL,
    FOREIGN KEY (instructor_id) REFERENCES instructors(instructor_id)
);

CREATE TABLE enrollments (
    student_id INTEGER NOT NULL,
    course_id INTEGER NOT NULL,
    semester TEXT NOT NULL,
    grade REAL CHECK (grade IS NULL OR (grade >= 0 AND grade <= 100)),
    PRIMARY KEY (student_id, course_id, semester),
    FOREIGN KEY (student_id) REFERENCES students(student_id) ON DELETE CASCADE,
    FOREIGN KEY (course_id) REFERENCES courses(course_id) ON DELETE CASCADE
);
""")
conn.commit()
print("Tables created.")

Tables created.


### 4A. Inspect the schema

`sqlite_master` is SQLite's metadata table. It records database objects such as tables and indexes. `PRAGMA table_info(table_name)` shows the columns of a particular table.

In [29]:
schema = cur.execute("""
SELECT name, sql
FROM sqlite_master
WHERE type = 'table' AND name NOT LIKE 'sqlite_%'
ORDER BY name
""").fetchall()

for table_name, create_sql in schema:
    print(f"--- {table_name} ---")
    print(create_sql, "\\n")

--- courses ---
CREATE TABLE courses (
    course_id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    credits INTEGER NOT NULL CHECK (credits BETWEEN 1 AND 6),
    instructor_id INTEGER NOT NULL,
    FOREIGN KEY (instructor_id) REFERENCES instructors(instructor_id)
) \n
--- enrollments ---
CREATE TABLE enrollments (
    student_id INTEGER NOT NULL,
    course_id INTEGER NOT NULL,
    semester TEXT NOT NULL,
    grade REAL CHECK (grade IS NULL OR (grade >= 0 AND grade <= 100)),
    PRIMARY KEY (student_id, course_id, semester),
    FOREIGN KEY (student_id) REFERENCES students(student_id) ON DELETE CASCADE,
    FOREIGN KEY (course_id) REFERENCES courses(course_id) ON DELETE CASCADE
) \n
--- instructors ---
CREATE TABLE instructors (
    instructor_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    department TEXT NOT NULL
) \n
--- students ---
CREATE TABLE students (
    student_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    age INTEGER CHECK (age >

## 5. Insert sample data

`INSERT INTO` adds rows. `executemany()` is convenient when inserting multiple records. The `?` placeholders are parameter markers; values are supplied separately instead of being pasted into SQL text.

`commit()` saves the transaction. If a transaction fails before commit, changes can usually be rolled back.

In [30]:
instructors = [
    (1, "Dr. Meera Shah", "Computer Science"),
    (2, "Prof. Arjun Rao", "Mathematics"),
    (3, "Dr. Sana Khan", "Data Science"),
]
students = [
    (1, "Aarav", "aarav@example.edu", 19, "Delhi"),
    (2, "Diya", "diya@example.edu", 20, "Noida"),
    (3, "Kabir", "kabir@example.edu", 18, "Delhi"),
    (4, "Anaya", "anaya@example.edu", 21, "Gurugram"),
    (5, "Ishaan", "ishaan@example.edu", 19, "Faridabad"),
    (6, "Mira", "mira@example.edu", 20, "Delhi"),
]
courses = [
    (101, "Database Systems", 4, 1),
    (102, "Python Programming", 3, 1),
    (103, "Discrete Mathematics", 4, 2),
    (104, "Data Analytics", 3, 3),
]
enrollments = [
    (1, 101, "2026-Sem1", 92),
    (1, 102, "2026-Sem1", 88),
    (2, 101, "2026-Sem1", 81),
    (2, 104, "2026-Sem1", 90),
    (3, 102, "2026-Sem1", 76),
    (3, 103, "2026-Sem1", 84),
    (4, 101, "2026-Sem1", 95),
    (4, 104, "2026-Sem1", 87),
    (5, 103, "2026-Sem1", 72),
    (6, 101, "2026-Sem1", 86),
    (6, 102, "2026-Sem1", 91),
    (6, 104, "2026-Sem1", 94),
]

cur.executemany("INSERT INTO instructors VALUES (?, ?, ?)", instructors)
cur.executemany("INSERT INTO students VALUES (?, ?, ?, ?, ?)", students)
cur.executemany("INSERT INTO courses VALUES (?, ?, ?, ?)", courses)
cur.executemany("INSERT INTO enrollments VALUES (?, ?, ?, ?)", enrollments)
conn.commit()
print("Inserted", cur.execute("SELECT COUNT(*) FROM students").fetchone()[0], "students")
print("Inserted", cur.execute("SELECT COUNT(*) FROM courses").fetchone()[0], "courses")
print("Inserted", cur.execute("SELECT COUNT(*) FROM enrollments").fetchone()[0], "enrollments")

Inserted 6 students
Inserted 4 courses
Inserted 12 enrollments


## 6. Read data: SELECT, WHERE, ORDER BY, LIMIT

- `SELECT` chooses the columns to display.
- `WHERE` filters rows.
- `ORDER BY` sorts results.
- `LIMIT` restricts how many rows are returned.
- Use explicit column names rather than `SELECT *` in production queries when possible, because explicit columns make the result easier to maintain.

In [ ]:
cur.execute("""
SELECT student_id, name, city, age
FROM students
WHERE city = ?
ORDER BY age DESC
""", ("Delhi",))
for row in cur.fetchall():
    print(row)

(6, 'Mira', 'Delhi', 20)
(1, 'Aarav', 'Delhi', 19)
(3, 'Kabir', 'Delhi', 18)


### 6A. Filter with multiple conditions

`AND` requires both conditions to be true. `OR` requires at least one condition to be true. `IN` matches any value in a list; `BETWEEN` checks an inclusive range; `LIKE` performs simple pattern matching.

In [32]:
rows = cur.execute("""
SELECT name, age, city
FROM students
WHERE age BETWEEN 19 AND 20
  AND city IN ('Delhi', 'Noida')
ORDER BY name
""").fetchall()
rows

[('Aarav', 19, 'Delhi'), ('Diya', 20, 'Noida'), ('Mira', 20, 'Delhi')]

## 7. CRUD: insert, read, update, delete

CRUD stands for **Create, Read, Update, Delete**—the four basic operations used by many applications.

Always use a `WHERE` clause for targeted `UPDATE` and `DELETE` statements. Without it, every row may be changed or removed.

In [ ]:
# CREATE: add one student.
cur.execute(
    "INSERT INTO students (student_id, name, email, age, city) VALUES (?, ?, ?, ?, ?)",
    (7, "Rohan", "rohan@example.edu", 20, "Delhi")
)
conn.commit()

# READ: verify the inserted record.
print(cur.execute("SELECT * FROM students WHERE student_id = ?", (7,)).fetchone())

(7, 'Rohan', 'rohan@example.edu', 20, 'Delhi')


In [34]:
# UPDATE: change one student's city.
cur.execute("UPDATE students SET city = ? WHERE student_id = ?", ("Jaipur", 7))
conn.commit()
print(cur.execute("SELECT student_id, name, city FROM students WHERE student_id = ?", (7,)).fetchone())

(7, 'Rohan', 'Jaipur')


In [35]:
# DELETE: remove the demonstration row.
cur.execute("DELETE FROM students WHERE student_id = ?", (7,))
conn.commit()
print("Remaining student count:", cur.execute("SELECT COUNT(*) FROM students").fetchone()[0])

Remaining student count: 6


## 8. Aggregation: COUNT, AVG, MIN, MAX, SUM, GROUP BY, HAVING

Aggregate functions summarize multiple rows:
- `COUNT(*)`: number of rows.
- `AVG(column)`: average of non-NULL values.
- `MIN` / `MAX`: smallest / largest value.
- `SUM(column)`: total of values.

`GROUP BY` creates one result per group. `HAVING` filters groups after aggregation; `WHERE` filters individual rows before grouping.

In [36]:
cur.execute("""
SELECT c.title,
       COUNT(e.student_id) AS enrolled_students,
       ROUND(AVG(e.grade), 2) AS average_grade,
       MIN(e.grade) AS lowest_grade,
       MAX(e.grade) AS highest_grade
FROM courses AS c
LEFT JOIN enrollments AS e ON e.course_id = c.course_id
GROUP BY c.course_id, c.title
ORDER BY average_grade DESC
""")
for row in cur.fetchall():
    print(row)

('Data Analytics', 3, 90.33, 87.0, 94.0)
('Database Systems', 4, 88.5, 81.0, 95.0)
('Python Programming', 3, 85.0, 76.0, 91.0)
('Discrete Mathematics', 2, 78.0, 72.0, 84.0)


### 8A. HAVING example

This query shows courses whose average grade is at least 85. `HAVING` is needed because the condition applies to an aggregate result (`AVG(grade)`), not an individual enrollment row.

In [37]:
cur.execute("""
SELECT c.title, ROUND(AVG(e.grade), 2) AS average_grade
FROM courses AS c
JOIN enrollments AS e ON e.course_id = c.course_id
GROUP BY c.course_id, c.title
HAVING AVG(e.grade) >= 85
ORDER BY average_grade DESC
""")
cur.fetchall()

[('Data Analytics', 90.33),
 ('Database Systems', 88.5),
 ('Python Programming', 85.0)]

## 9. Relational joins

A **JOIN** combines rows from tables using a related column.

- **INNER JOIN:** returns only matching rows from both sides.
- **LEFT JOIN:** returns every row from the left table and matching rows from the right; missing matches appear as `NULL`.
- **RIGHT/FULL OUTER JOIN:** supported by modern SQLite versions, but availability depends on SQLite version. For a beginner demo, INNER and LEFT JOIN cover the most common cases.
- A join condition should connect keys that represent the same relationship.

### 9A. INNER JOIN: enrollment report

This report joins enrollments to students and courses, so each row includes a student's name, course title, semester, and grade.

In [38]:
cur.execute("""
SELECT s.name AS student,
       c.title AS course,
       e.semester,
       e.grade
FROM enrollments AS e
JOIN students AS s ON s.student_id = e.student_id
JOIN courses AS c ON c.course_id = e.course_id
ORDER BY s.name, c.title
""")
for row in cur.fetchall():
    print(row)

('Aarav', 'Database Systems', '2026-Sem1', 92.0)
('Aarav', 'Python Programming', '2026-Sem1', 88.0)
('Anaya', 'Data Analytics', '2026-Sem1', 87.0)
('Anaya', 'Database Systems', '2026-Sem1', 95.0)
('Diya', 'Data Analytics', '2026-Sem1', 90.0)
('Diya', 'Database Systems', '2026-Sem1', 81.0)
('Ishaan', 'Discrete Mathematics', '2026-Sem1', 72.0)
('Kabir', 'Discrete Mathematics', '2026-Sem1', 84.0)
('Kabir', 'Python Programming', '2026-Sem1', 76.0)
('Mira', 'Data Analytics', '2026-Sem1', 94.0)
('Mira', 'Database Systems', '2026-Sem1', 86.0)
('Mira', 'Python Programming', '2026-Sem1', 91.0)


### 9B. LEFT JOIN: include courses with no enrollments

A LEFT JOIN is useful when you need to find entities that may have no related records. The following query lists all courses, including courses with zero enrollments.

In [39]:
cur.execute("""
SELECT c.title, COUNT(e.student_id) AS enrollment_count
FROM courses AS c
LEFT JOIN enrollments AS e ON e.course_id = c.course_id
GROUP BY c.course_id, c.title
ORDER BY c.course_id
""")
cur.fetchall()

[('Database Systems', 4),
 ('Python Programming', 3),
 ('Discrete Mathematics', 2),
 ('Data Analytics', 3)]

### 9C. Find students with no enrollment

When using a LEFT JOIN to find unmatched rows, filter for a NULL key on the right side. In this sample, all initial students have enrollments, so the result is empty. This is still a useful pattern for real systems.

In [40]:
cur.execute("""
SELECT s.student_id, s.name
FROM students AS s
LEFT JOIN enrollments AS e ON e.student_id = s.student_id
WHERE e.student_id IS NULL
""")
cur.fetchall()

[]

## 10. Many-to-many relationships and junction tables

A student can enroll in several courses, and each course can contain several students. Instead of storing multiple course IDs in one student row, we use the `enrollments` table.

The pair of foreign keys identifies the two related records. In this example, the composite primary key `(student_id, course_id, semester)` also prevents the same student from being enrolled in the same course twice in the same semester.

In [41]:
cur.execute("""
SELECT s.name, COUNT(e.course_id) AS courses_taken
FROM students AS s
LEFT JOIN enrollments AS e ON e.student_id = s.student_id
GROUP BY s.student_id, s.name
ORDER BY courses_taken DESC, s.name
""")
cur.fetchall()

[('Mira', 3),
 ('Aarav', 2),
 ('Anaya', 2),
 ('Diya', 2),
 ('Kabir', 2),
 ('Ishaan', 1)]

## 11. Subqueries

A subquery is a query nested inside another SQL statement. It can be used to compare a row against an aggregate or to filter by a derived result.

This example finds students whose grade in any course is above the overall average grade.

In [42]:
cur.execute("""
SELECT DISTINCT s.name
FROM students AS s
JOIN enrollments AS e ON e.student_id = s.student_id
WHERE e.grade > (SELECT AVG(grade) FROM enrollments)
ORDER BY s.name
""")
cur.fetchall()

[('Aarav',), ('Anaya',), ('Diya',), ('Mira',)]

## 12. CASE: create categories in a query

`CASE` is SQL's conditional expression. It can transform values into labels without changing the stored data—for example, grouping grades into performance bands.

In [43]:
cur.execute("""
SELECT s.name, c.title, e.grade,
       CASE
           WHEN e.grade >= 90 THEN 'Excellent'
           WHEN e.grade >= 80 THEN 'Good'
           WHEN e.grade >= 70 THEN 'Satisfactory'
           ELSE 'Needs improvement'
       END AS performance
FROM enrollments AS e
JOIN students AS s ON s.student_id = e.student_id
JOIN courses AS c ON c.course_id = e.course_id
ORDER BY e.grade DESC
LIMIT 10
""")
cur.fetchall()

[('Anaya', 'Database Systems', 95.0, 'Excellent'),
 ('Mira', 'Data Analytics', 94.0, 'Excellent'),
 ('Aarav', 'Database Systems', 92.0, 'Excellent'),
 ('Mira', 'Python Programming', 91.0, 'Excellent'),
 ('Diya', 'Data Analytics', 90.0, 'Excellent'),
 ('Aarav', 'Python Programming', 88.0, 'Good'),
 ('Anaya', 'Data Analytics', 87.0, 'Good'),
 ('Mira', 'Database Systems', 86.0, 'Good'),
 ('Kabir', 'Discrete Mathematics', 84.0, 'Good'),
 ('Diya', 'Database Systems', 81.0, 'Good')]

## 13. Parameterized queries and SQL injection

Never build SQL by concatenating untrusted input into a query string. Use placeholders (`?`) and pass values as a tuple. This keeps data separate from SQL instructions and helps prevent SQL injection.

Unsafe pattern (do not use):
`sql = "SELECT * FROM students WHERE name = '" + user_input + "'"`

Safe pattern:
`cur.execute("SELECT * FROM students WHERE name = ?", (user_input,))`

In [44]:
user_input = "Aarav"
safe_rows = cur.execute(
    "SELECT student_id, name, email FROM students WHERE name = ?",
    (user_input,)
).fetchall()
safe_rows

[(1, 'Aarav', 'aarav@example.edu')]

## 14. Transactions: commit and rollback

A transaction groups multiple changes into one unit of work. `commit()` makes the changes durable; `rollback()` discards uncommitted changes.

Transactions help preserve consistency when several related changes must succeed together. SQLite supports transactions, and Python's `sqlite3` connection manages them according to its transaction settings.

In [45]:
try:
    cur.execute(
        "INSERT INTO students (student_id, name, email, age, city) VALUES (?, ?, ?, ?, ?)",
        (99, "Temporary Student", "temporary@example.edu", 18, "Delhi")
    )
    print("Inside transaction:", cur.execute(
        "SELECT COUNT(*) FROM students WHERE student_id = 99"
    ).fetchone()[0])
    conn.rollback()
except sqlite3.Error as exc:
    conn.rollback()
    print("Transaction failed:", exc)

print("After rollback:", cur.execute(
    "SELECT COUNT(*) FROM students WHERE student_id = 99"
).fetchone()[0])

Inside transaction: 1
After rollback: 0


## 15. NULL and missing values

SQL uses `NULL` to represent missing or unknown information. It is not the same as zero or an empty string. Use `IS NULL` and `IS NOT NULL`; do not compare with `= NULL`.

Example: `WHERE grade IS NULL` finds enrollments whose grade has not been entered.

In [46]:
cur.execute("SELECT student_id, course_id FROM enrollments WHERE grade IS NULL")
cur.fetchall()

[]

## 16. Indexes and query performance

An index is an additional data structure that can speed up lookups and joins, much like an index in a book. Indexes consume storage and can make inserts/updates slower because the index must also be maintained.

Primary keys and UNIQUE constraints generally create indexes automatically. Add indexes to columns frequently used for filtering or joining after considering the workload.

In [47]:
cur.execute("CREATE INDEX IF NOT EXISTS idx_enrollments_course_id ON enrollments(course_id)")
conn.commit()

plan = cur.execute(
    "EXPLAIN QUERY PLAN SELECT * FROM enrollments WHERE course_id = ?",
    (101,)
).fetchall()
plan

[(3,
  0,
  0,
  'SEARCH enrollments USING INDEX idx_enrollments_course_id (course_id=?)')]

## 17. Data integrity: constraints in action

Constraints reject invalid data before it can corrupt the logical model. This example tries to insert an impossible age and catches the resulting integrity error.

In [48]:
try:
    cur.execute(
        "INSERT INTO students (student_id, name, email, age, city) VALUES (?, ?, ?, ?, ?)",
        (88, "Invalid Age", "invalid@example.edu", 12, "Delhi")
    )
    conn.commit()
except sqlite3.IntegrityError as exc:
    conn.rollback()
    print("Rejected by database constraint:", exc)

Rejected by database constraint: CHECK constraint failed: age >= 16


## 18. Export query results to CSV

SQL databases store data; CSV is a convenient interchange format for spreadsheets and analysis tools. Python can export query results using the standard-library `csv` module.

In [49]:
import csv

report = cur.execute("""
SELECT s.name, c.title, e.grade
FROM enrollments AS e
JOIN students AS s ON s.student_id = e.student_id
JOIN courses AS c ON c.course_id = e.course_id
ORDER BY s.name, c.title
""").fetchall()

csv_path = Path("enrollment_report.csv")
with csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["student", "course", "grade"])
    writer.writerows(report)

print("Exported:", csv_path.resolve())
print("Rows exported:", len(report))

Exported: /content/enrollment_report.csv
Rows exported: 12


## 19. Quick reference: daily SQL patterns

| Goal | SQL pattern |
|---|---|
| View all students | `SELECT * FROM students;` |
| Filter by city | `SELECT name FROM students WHERE city = ?;` |
| Sort by age | `SELECT name, age FROM students ORDER BY age DESC;` |
| Add a row | `INSERT INTO students (...) VALUES (...);` |
| Change a row | `UPDATE students SET city = ? WHERE student_id = ?;` |
| Delete a row | `DELETE FROM students WHERE student_id = ?;` |
| Count rows | `SELECT COUNT(*) FROM students;` |
| Group and summarize | `SELECT city, COUNT(*) FROM students GROUP BY city;` |
| Combine related tables | `... FROM students JOIN enrollments ON ...` |

Remember: use placeholders for values, inspect an UPDATE/DELETE with a matching SELECT first, and commit only when you intend to save the changes.

## 20. Close the connection

Close database connections when finished. In a larger program, use a context manager or structure connection handling with `try/finally` so connections are closed even if an error occurs.

In [50]:
cur.close()
conn.close()
print("Database connection closed.")

Database connection closed.


## References

- Python `sqlite3` documentation: https://docs.python.org/3/library/sqlite3.html
- SQLite documentation: https://www.sqlite.org/docs.html
- SQLite SQL language reference: https://www.sqlite.org/lang.html
- SQLite foreign keys: https://www.sqlite.org/foreignkeys.html
- SQLite SELECT statement: https://www.sqlite.org/lang_select.html